<a href="https://colab.research.google.com/github/wvb20/cv-face-alignment/blob/main/notebooks/03_cnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# === Cell 1: Bootstrap — Drive, repo, src/ imports ===
import os, sys, time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from google.colab import drive

drive.mount('/content/drive')

REPO_PATH = '/content/cv-face-alignment'
if not os.path.exists(REPO_PATH):
    !git clone https://github.com/wvb20/cv-face-alignment.git {REPO_PATH}
else:
    !cd {REPO_PATH} && git pull

if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

from src import config, data, evaluate, features, models, visualise, io
print('✓ Setup complete')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# === Cell 2: Load data, build train/val datasets ===
images, points = data.load_train()
train_idx, val_idx = data.get_split()

X_train, y_train = images[train_idx], points[train_idx]
X_val,   y_val   = images[val_idx],   points[val_idx]

train_ds = data.FaceLandmarksDataset(X_train, y_train, augment=True)
val_ds   = data.FaceLandmarksDataset(X_val,   y_val,   augment=False)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=64, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f'Train: {len(train_ds)} images ({len(train_loader)} batches)')
print(f'Val:   {len(val_ds)} images ({len(val_loader)} batches)')

# Quick sanity check — pull one batch and verify shapes
img_batch, pt_batch = next(iter(train_loader))
print(f'\nOne batch:')
print(f'  Images: {img_batch.shape}, dtype={img_batch.dtype}')
print(f'  Points: {pt_batch.shape}, dtype={pt_batch.dtype}')
print(f'  Image range: [{img_batch.min():.2f}, {img_batch.max():.2f}]  (post-normalisation)')
print(f'  Point range: [{pt_batch.min():.2f}, {pt_batch.max():.2f}]  (should be ~[-1, 1])')

In [ ]:
# === Cell 3: Visualise augmented batches to verify landmarks track correctly ===
# Make a fresh dataset so we can call __getitem__ multiple times to see different augs
aug_ds = data.FaceLandmarksDataset(X_train, y_train, augment=True)

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
np.random.seed(0)
torch.manual_seed(0)

for ax in axes.flat:
    idx = np.random.randint(len(aug_ds))
    img_t, pt_t = aug_ds[idx]
    # De-normalise image for display
    img_disp = img_t.numpy().transpose(1, 2, 0)
    for c in range(3):
        img_disp[..., c] = img_disp[..., c] * config.NORMALISE_STD[c] + config.NORMALISE_MEAN[c]
    img_disp = np.clip(img_disp, 0, 1)
    # De-normalise points
    pts_disp = models.points_to_pixels(pt_t.numpy(), config.IMAGE_SIZE)

    ax.imshow(img_disp)
    ax.scatter(pts_disp[:, 0], pts_disp[:, 1], s=80, c='lime',
               edgecolors='black', linewidths=1.5)
    ax.set_title(f'#{idx}')
    ax.axis('off')

fig.suptitle('Augmented training samples — landmarks should still land on features',
             fontsize=14)
fig.tight_layout()
plt.savefig(f'{config.FIGURES_DIR}/09_augmentation_check.png',
            dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# === Cell 4: Training and validation step functions ===

def train_one_epoch(model, loader, optimiser, criterion, device):
    model.train()
    total_loss = 0.0
    for imgs, pts in loader:
        imgs = imgs.to(device, non_blocking=True)
        pts  = pts.to(device, non_blocking=True)

        optimiser.zero_grad()
        preds = model(imgs)
        loss = criterion(preds, pts)
        loss.backward()
        optimiser.step()

        total_loss += loss.item() * imgs.size(0)
    return total_loss / len(loader.dataset)


@torch.no_grad()
def evaluate_model(model, loader, criterion, device):
    """Returns (mean_loss, all_preds_pixels, all_targets_pixels)."""
    model.eval()
    total_loss = 0.0
    all_preds, all_gts = [], []
    for imgs, pts in loader:
        imgs = imgs.to(device, non_blocking=True)
        pts  = pts.to(device, non_blocking=True)

        preds = model(imgs)
        loss = criterion(preds, pts)
        total_loss += loss.item() * imgs.size(0)

        # Convert back to pixel coordinates for NME calc
        preds_px = models.points_to_pixels(preds.cpu().numpy(),
                                            config.IMAGE_SIZE)
        gts_px   = models.points_to_pixels(pts.cpu().numpy(),
                                            config.IMAGE_SIZE)
        all_preds.append(preds_px)
        all_gts.append(gts_px)

    return (total_loss / len(loader.dataset),
            np.concatenate(all_preds),
            np.concatenate(all_gts))